# ☕ Coffee Sales Analysis

## Project Overview

This notebook supports the **Coffee Sales Analytics Dashboard** built in Power BI.

The Power BI report uses a table named `Coffe_sales` and analyzes fields including:

- `coffee_name`
- `money`
- `date`
- `Month_name`
- `cash_type`
- `card`

The Power BI model also contains measures such as:

- `Total_Revenue`
- `Target_Revenue`
- `Types_coffee`

This notebook focuses on reproducible data cleaning, exploratory analysis, revenue trends, product performance, and payment-pattern analysis.


## 1. Import Libraries


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("data/coffee_sales.csv")


## 2. Load the Dataset

Export the coffee-sales source data from Power BI / Power Query or use the original source dataset and save it as:

`data/coffee_sales.csv`


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Add the coffee-sales CSV as data/coffee_sales.csv "
        "and rerun the notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


## 3. Data Understanding


In [ ]:
print("Columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:>2}. {col}")

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
missing = (
    df.isna().sum()
      .to_frame("missing_count")
      .assign(
          missing_pct=lambda x: (
              x["missing_count"] / len(df) * 100
          ).round(2)
      )
      .sort_values("missing_pct", ascending=False)
)

display(missing)


## 4. Data Cleaning


In [ ]:
clean = df.copy()

# Standardize column names without changing the original business meaning.
clean.columns = [c.strip() for c in clean.columns]

# Remove exact duplicate records.
clean = clean.drop_duplicates()

# Trim text fields.
for col in clean.select_dtypes(include="object").columns:
    clean[col] = clean[col].astype(str).str.strip().replace({"nan": np.nan})

# Convert date.
if "date" in clean.columns:
    clean["date"] = pd.to_datetime(clean["date"], errors="coerce")

# Convert revenue / transaction amount.
if "money" in clean.columns:
    clean["money"] = pd.to_numeric(clean["money"], errors="coerce")

# Re-create Month_name from date when necessary.
if "date" in clean.columns:
    if "Month_name" not in clean.columns:
        clean["Month_name"] = clean["date"].dt.month_name()
    clean["Month_Number"] = clean["date"].dt.month

print("Cleaned shape:", clean.shape)
display(clean.head())


## 5. Key Sales KPIs


In [ ]:
if "money" not in clean.columns:
    raise ValueError("The expected 'money' column is missing.")

total_revenue = clean["money"].sum()
transactions = len(clean)
avg_transaction = clean["money"].mean()

coffee_types = (
    clean["coffee_name"].nunique()
    if "coffee_name" in clean.columns
    else np.nan
)

kpis = pd.DataFrame({
    "KPI": [
        "Total Revenue",
        "Transactions",
        "Average Transaction Value",
        "Coffee Types"
    ],
    "Value": [
        total_revenue,
        transactions,
        avg_transaction,
        coffee_types
    ]
})

display(kpis)


## 6. Revenue by Coffee Product


In [ ]:
if "coffee_name" in clean.columns:
    product_revenue = (
        clean.groupby("coffee_name", dropna=False)["money"]
             .agg(["sum", "count", "mean"])
             .rename(columns={
                 "sum": "Revenue",
                 "count": "Transactions",
                 "mean": "Avg_Transaction"
             })
             .sort_values("Revenue", ascending=False)
    )

    display(product_revenue)

    top = product_revenue["Revenue"].head(10).sort_values()

    plt.figure(figsize=(9, 5))
    plt.barh(top.index.astype(str), top.values)
    plt.title("Top Coffee Products by Revenue")
    plt.xlabel("Revenue")
    plt.tight_layout()
    plt.show()
else:
    print("coffee_name column not found.")


## 7. Monthly Revenue Trend


In [ ]:
if {"date", "money"}.issubset(clean.columns):
    monthly_revenue = (
        clean.dropna(subset=["date"])
             .set_index("date")["money"]
             .resample("ME")
             .sum()
    )

    display(monthly_revenue.to_frame("Revenue"))

    plt.figure(figsize=(10, 4))
    plt.plot(
        monthly_revenue.index,
        monthly_revenue.values,
        marker="o"
    )
    plt.title("Monthly Revenue Trend")
    plt.xlabel("Month")
    plt.ylabel("Revenue")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("date and money columns are required for trend analysis.")


## 8. Revenue by Month Name


In [ ]:
if {"Month_name", "money"}.issubset(clean.columns):
    if "Month_Number" in clean.columns:
        month_order = (
            clean[["Month_name", "Month_Number"]]
            .dropna()
            .drop_duplicates()
            .sort_values("Month_Number")["Month_name"]
            .tolist()
        )
    else:
        month_order = list(pd.date_range("2025-01-01", periods=12, freq="MS").month_name())

    by_month = (
        clean.groupby("Month_name")["money"]
             .sum()
             .reindex(month_order)
             .dropna()
    )

    display(by_month.to_frame("Revenue"))

    plt.figure(figsize=(10, 4))
    plt.bar(by_month.index, by_month.values)
    plt.title("Revenue by Month")
    plt.xlabel("Month")
    plt.ylabel("Revenue")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 9. Payment Method Analysis

The Power BI model contains `cash_type` and `card`, so this section analyzes those fields when they are present in the exported data.


In [ ]:
if "cash_type" in clean.columns:
    payment_summary = (
        clean.groupby("cash_type", dropna=False)["money"]
             .agg(["sum", "count"])
             .rename(columns={"sum": "Revenue", "count": "Transactions"})
             .sort_values("Revenue", ascending=False)
    )

    display(payment_summary)

    plt.figure(figsize=(7, 4))
    plt.bar(
        payment_summary.index.astype(str),
        payment_summary["Revenue"]
    )
    plt.title("Revenue by Payment Type")
    plt.xlabel("Payment Type")
    plt.ylabel("Revenue")
    plt.tight_layout()
    plt.show()
else:
    print("cash_type column not found.")


In [ ]:
if "card" in clean.columns:
    card_summary = (
        clean.groupby("card", dropna=False)["money"]
             .agg(["sum", "count"])
             .rename(columns={"sum": "Revenue", "count": "Transactions"})
             .sort_values("Revenue", ascending=False)
             .head(15)
    )

    display(card_summary)


## 10. Daily and Weekly Sales Patterns


In [ ]:
if "date" in clean.columns:
    clean["Day_Name"] = clean["date"].dt.day_name()
    clean["Day_Number"] = clean["date"].dt.dayofweek

    daily = (
        clean.groupby(["Day_Number", "Day_Name"])["money"]
             .agg(["sum", "count"])
             .reset_index()
             .sort_values("Day_Number")
    )

    display(daily)

    plt.figure(figsize=(8, 4))
    plt.bar(daily["Day_Name"], daily["sum"])
    plt.title("Revenue by Day of Week")
    plt.xlabel("Day")
    plt.ylabel("Revenue")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


## 11. Revenue Target Analysis

`Target_Revenue` is a measure in the Power BI model.

Its DAX formula is not available from the report layout alone, so this notebook does **not invent** a target calculation.

If the exported dataset contains a real target column, the following code will evaluate target achievement.


In [ ]:
target_candidates = [
    c for c in clean.columns
    if c.lower() in {
        "target_revenue",
        "revenue_target",
        "target"
    }
]

if target_candidates:
    target_col = target_candidates[0]
    clean[target_col] = pd.to_numeric(clean[target_col], errors="coerce")

    actual = clean["money"].sum()
    target = clean[target_col].sum()

    target_result = pd.DataFrame({
        "Metric": ["Actual Revenue", "Target Revenue", "Achievement %"],
        "Value": [
            actual,
            target,
            (actual / target * 100) if target else np.nan
        ]
    })

    display(target_result)
else:
    print(
        "No source target column found. "
        "Use the real Power BI Target_Revenue measure for target analysis."
    )


## 12. Business Insight Summary

After running this notebook on the real source data, summarize the actual results:

- Highest-revenue coffee product
- Lowest-performing coffee product
- Best and weakest sales months
- Most-used payment method
- Average transaction value
- Revenue trend over time
- Any visible seasonal pattern
- Actual-vs-target performance, if the real target measure/data is available

Do not write conclusions until the notebook has been executed on the real dataset.


## 13. Export Cleaned Data for Power BI


In [ ]:
OUTPUT_DIR = Path("cleaned_data")
OUTPUT_DIR.mkdir(exist_ok=True)

output_path = OUTPUT_DIR / "coffee_sales_cleaned.csv"
clean.to_csv(output_path, index=False)

print("Saved:", output_path)
